<h3>creating spark object which can be used for upcoming objectives</h3>

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark=SparkSession.builder.appName('assingment_06').getOrCreate()

In [ ]:
df = spark.read.csv(
    str(project_root / "dataset" / "SampleSuperstore.csv"),
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

In [6]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



<h3> basic info of df and renaming some columns</h3>

In [7]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [8]:
df.count()

9994

In [9]:
len(df.columns)


21

In [10]:
cl=df.columns
print(cl)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [11]:
from pyspark.sql.functions import col

for c in cl:
    print(c, df.filter(col(c).isNull()).count())

Row ID 0
Order ID 0
Order Date 0
Ship Date 0
Ship Mode 0
Customer ID 0
Customer Name 0
Segment 0
Country 0
City 0
State 0
Postal Code 0
Region 0
Product ID 0
Category 0
Sub-Category 0
Product Name 0
Sales 0
Quantity 0
Discount 0
Profit 0


In [12]:
df=df.withColumnRenamed('sales','total_sales')
df=df.withColumnRenamed('Profit','Net_Profit')


In [13]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'total_sales',
 'Quantity',
 'Discount',
 'Net_Profit']

<h3> Not using these as i have to again import the dataset and have to go with the whole check process again</h3>

In [14]:
# from pyspark.sql.types import StructType,StructField,IntegerType,DoubleType

In [15]:
# schema=StructType([
#     StructField('Quantity',IntegerType(),True),
#     StructField('Discount',DoubleType(),True)
# ])

<h3> change dtyppe on existing data frame</h3>

In [16]:
df=(df.withColumn("Quantity",col("Quantity").cast('integer'))
    .withColumn("Discount",col('Discount').cast('double')))

In [17]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Net_Profit: double (nullable = true)



<h3>Creating a temp view which i will use to create new columns inside my df that's why using df=.... to replace the current with updated and using sql queries for creating one , usinbg cast to cahnge data type as it wont compare string with integer or double </h3>

In [18]:
df.createOrReplaceTempView("tempo")

In [19]:
df = spark.sql("""
SELECT *,
       CASE
           WHEN CAST(total_sales AS DOUBLE) > 500
               THEN CAST(total_sales AS DOUBLE) * 0.18
           ELSE CAST(total_sales AS DOUBLE) * 0.10
       END AS tax,

       Net_Profit - (CAST(total_sales AS DOUBLE) * 0.18) AS profit_after_tax,
       CASE
            WHEN CAST(total_sales AS DOUBLE) > 1000 THEN 'YES'
            ELSE 'NO'
        END AS high_sales
FROM tempo
""")

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|total_sales|Quantity|Discount|Net_Profit|               tax|   profit_after_tax|high_sales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Secon

In [20]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'total_sales',
 'Quantity',
 'Discount',
 'Net_Profit',
 'tax',
 'profit_after_tax',
 'high_sales']

In [21]:
from pyspark.sql.functions import col

df = (
    df.withColumn("Quantity", col("Quantity").cast("integer"))
      .withColumn("Discount", col("Discount").cast("double"))
      .withColumn("total_sales", col("total_sales").cast("double"))
      .withColumn("Net_Profit", col("Net_Profit").cast("double"))
)

In [22]:
df.createOrReplaceTempView('tempo')

In [23]:
fl1 = spark.sql("""
SELECT *
FROM tempo
WHERE Net_Profit < 200
""")

fl1.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|total_sales|Quantity|Discount|Net_Profit|               tax|   profit_after_tax|high_sales|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|     1|CA-2016-152156

In [24]:
fl2=df.groupBy(df.Region).count()
print(fl2)
fl2.show()

DataFrame[Region: string, count: bigint]
+-------+-----+
| Region|count|
+-------+-----+
|  South| 1620|
|Central| 2323|
|   East| 2848|
|   West| 3203|
+-------+-----+



In [25]:
fl3=df.filter(df['Category']=='Office Supplies')
fl3.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|total_sales|Quantity|Discount|Net_Profit|               tax|   profit_after_tax|high_sales|
+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|     3|CA-2016-138688| 6/12/20

In [26]:
sel=df.select('Country').distinct()
sel.show()

+-------------+
|      Country|
+-------------+
|United States|
+-------------+



In [27]:
sel=df.select('sub-category').distinct()
sel.show()

+------------+
|sub-category|
+------------+
|   Envelopes|
|         Art|
|      Chairs|
| Furnishings|
|    Supplies|
|   Fasteners|
|     Binders|
|   Bookcases|
|      Labels|
|       Paper|
| Accessories|
|     Copiers|
|      Phones|
|    Machines|
|     Storage|
|  Appliances|
|      Tables|
+------------+



<h1>Finalized DataFrame after adding required columns and filtering data and getting multiple info and grouping</h1>

In [28]:
df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|total_sales|Quantity|Discount|Net_Profit|               tax|   profit_after_tax|high_sales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+----------+------------------+-------------------+----------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Secon

In [29]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Net_Profit: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- profit_after_tax: double (nullable = true)
 |-- high_sales: string (nullable = false)



<h3> Saving this updated dataframe in csv and parquet format </h3>

In [31]:
df.write.mode("overwrite").option("header", True).csv(str(project_root / "output" / "csv"))
df.write.mode("overwrite").parquet(str(project_root / "output" / "parquet"))
